Full-sky neutrino fluxes with `solve_fullsky`
=============================================

This notebook walks through `MCEqRun.solve_fullsky`, which propagates a
single primary spectrum through every (zenith, azimuth) pixel of a sky
grid in **one multi-RHS solve**. Three things to know:

1. **Stage-5 LPT carousel is the only multi-RHS path.** It packs the
   K_total pixels into a `K_pipe`-column pipeline; when a column
   finishes its path the next pending pixel is loaded into that slot
   on the same step. Total kernel work is `Σ_pixels nsteps × ms/RHS`,
   not `max(nsteps) × K_total × ms/RHS`. Stage-3 zero-pad and Stage-4
   bucketing are gone — you don't need to choose. Works on all four
   backends (`numpy_etd2`, `cuda_etd2`, `mkl_etd2`,
   `accelerate_etd2`).

2. **The default atmosphere is NRLMSIS 2.1** (pure-Python, vectorised,
   fork-safe). The legacy NRLMSISE-00 backend is still available via
   `density_model=('MSIS00', …)` for back-compat.

3. **Geomagnetic rigidity cutoff is on by default** for MSIS-based and
   location-tagged CORSIKA atmospheres. The first call runs `gtracr`
   to build the per-pixel R_c map and caches it in MCEq's data
   directory (`<MCEq.data_dir>/gtracr_cutoffs/`); subsequent runs at
   the same detector hit the cache. Toggle with
   `MCEqRun(geomagnetic_cutoff=…)` or
   `solve_fullsky(geomagnetic_cutoff=…)`.

Imports.

In [ ]:
import time

import crflux.models as pm
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from MCEq.core import MCEqRun

Instantiate `MCEqRun` with the `MSIS21_IC` short-key (location-centered
NRLMSIS 2.1 at the South Pole). At the South Pole the atmosphere column
is azimuth-symmetric, but the full-sky machinery still exercises
K_total = n_zen × n_az distinct columns. Replace with
`('MSIS21_KM3NeT', ('ORCA', 'January'))` for a mid-latitude detector
with genuine azimuthal variation.

In [ ]:
mceq = MCEqRun(
    interaction_model='SIBYLL23D',
    primary_model=(pm.HillasGaisser2012, 'H3a'),
    theta_deg=0.0,
    density_model=('MSIS21_IC', ('SouthPole', 'January')),
    # geomagnetic_cutoff=None (default) auto-detects from the atmosphere.
    # For an MSIS-based detector-centered atmosphere it resolves to True.
)
print(f'dim_states = {mceq.dim_states}, n_E = {mceq.e_grid.size}')
print(f'geomagnetic_cutoff (auto) → '
      f'{mceq._is_geomag_eligible_atmosphere()}')

Define the (zenith, azimuth) grid. A 10° step (19 × 36 = 684 pixels)
runs comfortably on `numpy_etd2`. The same call scales to the 5° /
2664-pixel grid on `cuda_etd2` / `mkl_etd2` / `accelerate_etd2` in a
few minutes.

In [ ]:
step_deg = 10.0
zen_centres = np.linspace(0.0, 180.0, int(round(180.0 / step_deg)) + 1)
az_centres = np.arange(int(round(360.0 / step_deg))) * step_deg
n_zen, n_az = zen_centres.size, az_centres.size
K = n_zen * n_az
print(f'grid: {n_zen} zen × {n_az} az = K = {K} pixels')

E_target_GeV = np.array([1.0, 10.0, 100.0])

Baseline: serial pixel scan
---------------------------

The classic per-pixel loop — `set_zenith_azimuth(zen, az)` + `solve()`.
Sub-sampled grid so the baseline finishes in a minute or two on
`numpy_etd2`. With the auto-cutoff active on the run, we explicitly
disable it here to keep the baseline matching the no-cutoff
atmosphere-only case (the same flag is also accepted by `solve_fullsky`
for the same reason).

In [ ]:
zen_sub = zen_centres[::3]
az_sub = az_centres[::9]
K_sub = zen_sub.size * az_sub.size

log_E = np.log(mceq.e_grid)
log_targets = np.log(E_target_GeV)
numu_sub = np.zeros((E_target_GeV.size, zen_sub.size, az_sub.size))

t0 = time.time()
with tqdm(total=K_sub, desc='serial scan') as bar:
    for i_z, zen in enumerate(zen_sub):
        for i_a, az in enumerate(az_sub):
            mceq.set_zenith_azimuth(float(zen), float(az))
            mceq.solve()
            flux = (mceq.get_solution('total_numu', 0)
                    + mceq.get_solution('total_antinumu', 0))
            numu_sub[:, i_z, i_a] = np.exp(
                np.interp(log_targets, log_E,
                          np.log(np.maximum(flux, 1e-300)))
            )
            bar.update()
t_serial = time.time() - t0
print(f'serial wall: {t_serial:.1f}s '
      f'({t_serial / K_sub * 1e3:.1f} ms / pixel)')

One-shot `solve_fullsky` (carousel)
-----------------------------------

`solve_fullsky` builds a per-pixel integration path, packs them with
the LPT scheduler, and runs one multi-RHS dispatch. `carousel_K=None`
(default) resolves to `min(K, 128)` — the sweet spot from the
bench page. Pass `carousel_K=N` to override.

Because the atmosphere is MSIS-based and `geomagnetic_cutoff` defaults
to auto-on, the **first call builds the gtracr R_c map and caches it**
under MCEq's data directory (a `tqdm` bar shows progress). Subsequent
runs at the same detector hit the cache.

For this notebook we disable the cutoff so the baseline serial scan
above (no cutoff) and the multi-RHS solve are comparing the same
physics. Toggle to `True` to see the cutoff-applied skymap.

In [ ]:
t0 = time.time()
sol, nsteps_per_col = mceq.solve_fullsky(
    zen_centres, az_centres,
    geomagnetic_cutoff=False,   # match the serial baseline (no cutoff)
)
t_carousel = time.time() - t0
print(f'solve_fullsky (carousel): {t_carousel:.2f}s for K = {K} pixels')
print(f'  nsteps per pixel: min={nsteps_per_col.min()} '
      f'mean={nsteps_per_col.mean():.0f} max={nsteps_per_col.max()}')
print(f'  sol shape = {sol.shape}')
print(f'  speedup vs serial sub-grid extrapolated to K={K}: '
      f'×{(t_serial / K_sub) * K / t_carousel:.1f}')

Per-pixel flux extraction
-------------------------

`solve_fullsky` returns the full `(dim_states, K)` state. To turn it
into a `(n_E_target, n_zen, n_az)` skymap, sum particle + antiparticle
slices and interpolate to each target energy. log_target is scalar so
the bracketing index + weight are constant across pixels; one
vectorised lerp does the whole K-axis.

In [ ]:
pname2pref = mceq.pman.pname2pref
numu_pixel_grid = np.zeros((mceq.e_grid.size, K), dtype=np.float64)
for base in ('numu', 'antinumu'):
    for variant in (base, base + '_l', base + '_r'):
        if variant in pname2pref:
            p = pname2pref[variant]
            numu_pixel_grid += sol[p.lidx:p.uidx, :]

log_flux = np.log(np.maximum(numu_pixel_grid, 1e-300))
j = int(np.clip(np.searchsorted(log_E, log_targets[:, None]),
                1, log_E.size - 1)[0])
# Per-energy: scalar bracket + scalar weight, then K-wide lerp
numu_at_E = np.empty((E_target_GeV.size, K), dtype=np.float64)
for i, lt in enumerate(log_targets):
    ji = int(np.clip(np.searchsorted(log_E, lt, side='right'),
                     1, log_E.size - 1))
    w = (lt - log_E[ji - 1]) / (log_E[ji] - log_E[ji - 1])
    numu_at_E[i] = np.exp((1.0 - w) * log_flux[ji - 1] + w * log_flux[ji])

numu_at_E = numu_at_E.reshape(E_target_GeV.size, n_zen, n_az)
print(f'skymap shape = {numu_at_E.shape}')

Plot the zenith dependence at the three target energies, averaging
over azimuth. Overlay the serial-baseline sub-grid as crosses to
verify that the multi-RHS solve reproduces the per-pixel scan.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=False)
cos_theta = np.cos(np.deg2rad(zen_centres))
cos_theta_sub = np.cos(np.deg2rad(zen_sub))
for i, E in enumerate(E_target_GeV):
    ax = axes[i]
    ax.semilogy(cos_theta, numu_at_E[i].mean(axis=1), 'k-', lw=1.5,
                label='solve_fullsky (azimuth-averaged)')
    ax.plot(cos_theta_sub, numu_sub[i].mean(axis=1), 'r+', ms=8,
            label='serial scan' if i == 0 else None)
    ax.set_xlabel(r'$\cos\theta$')
    ax.set_title(rf'$E_{{\nu_\mu}} = {E:g}$ GeV')
axes[0].set_ylabel(
    r'$\Phi_{\nu_\mu + \bar\nu_\mu}$ [(cm$^2$ s sr GeV)$^{-1}$]'
)
axes[0].legend(loc='best', fontsize='small', frameon=False)
plt.tight_layout()

Per-pixel initial spectrum
--------------------------

Pass a 2-D `phi0` of shape `(dim_states, K)` to apply per-pixel
modifications to the primary directly. `solve_fullsky` recognises
2-D `phi0` and skips the auto-cutoff (the caller is in charge of the
per-pixel primary spectrum then).

The simplest use case is the geomagnetic cutoff — but you don't need
to build the 2-D phi0 yourself for that, just leave
`geomagnetic_cutoff=None` (or `True`) and MCEq does it for you. The
2-D phi0 hook is useful for anything else that varies the primary
per pixel: a non-standard composition model, a directional veto
mask, an anisotropic source, etc.

Below we demonstrate by zeroing the primary in the upgoing
hemisphere — purely to exercise the 2-D phi0 API.

In [ ]:
phi0_1d = mceq._phi0.copy()
phi0_2d = np.broadcast_to(phi0_1d[:, None],
                          (mceq.dim_states, K)).copy()
# Zero the upgoing half of the sky (zen > 90°), demonstrating
# per-pixel phi0.
upgoing_mask = (zen_centres[:, None] > 90.0) & np.ones_like(
    az_centres[None, :], dtype=bool
)
phi0_2d[:, upgoing_mask.ravel()] = 0.0

sol_masked, _ = mceq.solve_fullsky(
    zen_centres, az_centres, phi0=phi0_2d,
)
print(f'masked solve sol shape = {sol_masked.shape}')

Geomagnetic rigidity cutoff
---------------------------

For a real all-sky analysis at a mid-latitude detector, just leave
`geomagnetic_cutoff` on its default. The first call runs gtracr;
subsequent calls hit the cache.

In [ ]:
# Switch to ORCA (mid-latitude → strong azimuthal R_c structure).
mceq_orca = MCEqRun(
    interaction_model='SIBYLL23D',
    primary_model=(pm.HillasGaisser2012, 'H3a'),
    theta_deg=0.0,
    density_model=('MSIS21_KM3NeT', ('ORCA', 'January')),
)
# First call: builds + caches the R_c map (tqdm bar visible).
# Subsequent calls hit the cache.
sol_orca, _ = mceq_orca.solve_fullsky(
    zen_centres, az_centres,
    cutoff_kwargs={'iter_num': 5000},  # smaller MC for the demo
)
print(f'ORCA sol shape = {sol_orca.shape}')

**Cache management**: the cutoff map lives at
`<MCEq.data_dir>/gtracr_cutoffs/gtracr_cutoffs_<location>_<bfield>_v<N>_<hash>.npz`.
The key encodes `(location, B-field type, IGRF date, particle altitude,
MC iter count, rigidity scan, cache version)`. Bump
`MCEq.geometry.gtracr_cutoff.CACHE_VERSION` to invalidate every
existing cache file.